# VCC 2026 submission: public-screen effect borrowing

Builds the submission for whichever phase the downloaded controls bundle is for (final phase: contexts D, E, F; validation: A, B, C) from the method in `models/public_screens/` (RESULTS.md Round 4; preset c1_dev by default (c1_wide with each target's own signs on the wide genes, see models/submission_2026/README.md); c1_wide is the per-target predictions of c1_calibrated plus the cross-line shared knockdown response called on ~8,000 genes, which fixes the -1.26 direction-fidelity score of the first submission; c4_wide also amplifies the per-target part 4x) and submits it with the official `vcc` CLI.

* Each target in the bundle's `pert_counts.csv` gets 400 predicted cells per context: real control cells from that context with their counts scaled gene by gene by the predicted fold change (binomial thinning down, added Poisson counts up), so values stay raw integer counts as the 2026 scorer requires.
* Predicted change = H1 shared response + similarity-weighted H1 effects of other genes (similarity from the Replogle 2022 K562 genome-wide screen) + the target's own K562 profile, gated to genes detected in >= 1% of the context's controls; the target drops to its K562 remaining fraction.
* `vcc prep` needs ~25 GB RAM for a full submission, so `package_vcc.py` runs the official prep check per context and on the combined labels, then packages the `.vcc` exactly as prep does. On a High-RAM runtime you can run plain `vcc prep` instead (last cell).

The API key is asked for at runtime and only kept in this runtime's environment; it is never written to Drive or the repo.

**Build-only mode.** Set `BUILD_ONLY = True` in the build cell to rebuild a preset without spending a daily submission: the submit cell is skipped and the `.vcc` (about 3.1 GB) is saved with a small README of the preset and settings to `DRIVE_FOLDER` in your Google Drive, ready to share by link. The build is seeded, so the same preset and commit give the same predictions. The key is still needed, because the controls bundle downloads through `vcc`.

**Final phase.** Set `DATASET = 'controls-final'` (the default) in the data cell. The contexts and the target list are read from the bundle's `manifest.json` and `pert_counts.csv`, so nothing is assumed about A/B/C. The final phase needs `vcc` 0.3.0 or newer (installed in the setup cell), shows no scores until the winners are announced, allows two submissions per team per UTC day, and **only the team's last final entry counts**, so submit the preset you want judged last.

Runtime: standard CPU, ~40 GB free disk; about 20 minutes to build plus upload time. The whole Replogle K562 screen ships in the repo (`models/submission_2026/data/k562_gwps_all/`, rebuilt by `k562_pack.py` in the data cell), so any target list is covered and nothing is downloaded from figshare.

In [ ]:
!pip -q install anndata h5py scipy pandas
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']
!uv tool install --upgrade vcc-cli && vcc --version
import subprocess
_v = subprocess.run(['vcc', '--version'], capture_output=True, text=True).stdout.split()[1]
assert tuple(map(int, _v.split('.')[:2])) >= (0, 3), f'vcc {_v} is too old for the final phase: needs 0.3.0 or newer'
!git clone -q --depth 1 https://github.com/KunaalAgarwal/virtual_cell_notebooks /content/vcn
WORK = '/content/vcc2026'; os.makedirs(WORK, exist_ok=True); os.chdir(WORK)

## API key (asked once, kept only in this runtime)

In [ ]:
import getpass
os.environ['VCC_TOKEN'] = getpass.getpass('VCC API key: ')
!vcc whoami

## Data: the controls bundle for the current phase, and the Replogle K562 screen (shipped in the repo, no download)

In [ ]:
DATASET = 'controls-final'   # final phase (contexts D, E, F); 'controls' was the validation bundle (A, B, C)
EXPECT = {'controls-final': ['D', 'E', 'F'], 'controls': ['A', 'B', 'C']}[DATASET]
import json, shutil
shutil.rmtree(f'{WORK}/data', ignore_errors=True)   # never mix files from the other phase's bundle
!vcc datasets download {DATASET} -o {WORK}/{DATASET}.zip
!unzip -o -j -q {WORK}/{DATASET}.zip -d {WORK}/data
!ls -la {WORK}/data
M = '/content/vcn/models'
!python {M}/submission_2026/bundle.py {WORK}/data
manifest = json.load(open(f'{WORK}/data/manifest.json'))
assert sorted(manifest['contexts']) == EXPECT, f"bundle is for contexts {manifest['contexts']}, expected {EXPECT}: check DATASET"
print('bundle OK:', manifest.get('panel_id'), manifest['contexts'])

# Replogle 2022 K562 genome-wide screen, bulk z-scores (figshare article 20029387, file 35773217,
# which figshare has failed to serve from Colab). All 11,258 rows ship in the repo in compressed
# parts; this rebuilds the .h5ad (about 370 MB, 1 minute). Rows of the validation targets, the H1
# knockdowns and the controls are exact; the rest are within 0.0005 z of the original.
K562 = f'{WORK}/K562_gwps_rebuilt.h5ad'
K562_MIN = 300_000_000
!python {M}/submission_2026/k562_pack.py unpack --parts {M}/submission_2026/data/k562_gwps_all --out {K562}

def check_hdf5(path, min_bytes):
    ok = os.path.exists(path) and os.path.getsize(path) >= min_bytes and open(path, 'rb').read(8) == b'\x89HDF\r\n\x1a\n'
    size = os.path.getsize(path) if os.path.exists(path) else 0
    print(f'{"OK" if ok else "BAD"}: {path} ({size:,} bytes, {"HDF5 signature" if ok else "no HDF5 signature or too small"})')
    return ok

assert check_hdf5(K562, K562_MIN), 'K562 file missing or incomplete: re-run the setup cell (fresh clone of main) and this cell'
print('K562 file OK')


## Build the prediction (streams to disk; ~3 GB RAM)

In [ ]:
PRESET = 'c1_dev'   # or 'd3_dev' (K562 profile x3, no H1 terms). Final phase: only the team's LAST final entry counts
# earlier validation scores: 'c1_calibrated' (-0.1945), 'c1_wide' (-0.0295)
BUILD_ONLY = False   # True: skip vcc submit and save the .vcc + README to Google Drive instead
DRIVE_FOLDER = 'VCC2026_builds'   # folder under My Drive used when BUILD_ONLY is True
assert check_hdf5(K562, K562_MIN), 'the K562 file is missing or incomplete: re-run the data cell'
assert sorted(json.load(open(f'{WORK}/data/manifest.json'))['contexts']) == EXPECT, 'data folder is from the wrong phase: re-run the data cell'
!python {M}/submission_2026/build_submission.py --data {WORK}/data \
    --h1 {M}/submission_2026/data/h1_pseudobulk.npz --k562 {K562} \
    --method-dir {M}/public_screens --out {WORK}/pred.h5ad --diag {WORK}/diag --preset {PRESET}
assert os.path.exists(f'{WORK}/pred.h5ad'), 'build did not produce pred.h5ad: read the error above before packaging'

## Validate with the official CLI and package the .vcc

In [ ]:
PY = os.path.expanduser('~/.local/share/uv/tools/vcc-cli/bin/python')
!{PY} {M}/submission_2026/package_vcc.py --pred {WORK}/pred.h5ad --data {WORK}/data --out {WORK}/pred.vcc --tmp {WORK}

## Submit (counts toward the daily limit of two; skipped when `BUILD_ONLY` is True)

In [ ]:
if BUILD_ONLY:
    print("BUILD_ONLY is True: not submitting. Run the next cell to save the file to Drive.")
else:
    !vcc submit {WORK}/pred.vcc -m "screen-borrow {PRESET} {DATASET}" --wait


## Build-only: save the .vcc and a README to Google Drive (runs only when `BUILD_ONLY` is True)

Copying 3.1 GB to Drive takes a few minutes; the cell waits for the upload to finish before unmounting. Free Drive is 15 GB, so delete old builds you no longer need.

In [ ]:
import datetime, hashlib, json, shutil, subprocess
if not BUILD_ONLY:
    print("BUILD_ONLY is False: nothing saved to Drive.")
else:
    from google.colab import drive
    drive.mount('/content/drive')
    stamp = datetime.datetime.utcnow().strftime('%Y-%m-%d_%H%M')
    dest = f'/content/drive/MyDrive/{DRIVE_FOLDER}/{PRESET}_{stamp}'
    os.makedirs(dest, exist_ok=True)
    vcc_file = f'{WORK}/pred.vcc'
    h = hashlib.sha256()
    with open(vcc_file, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 24), b''):
            h.update(chunk)
    run = json.load(open(f'{WORK}/pred.json'))
    commit = subprocess.run(['git', '-C', '/content/vcn', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
    branch = subprocess.run(['git', '-C', '/content/vcn', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip()
    cli = subprocess.run(['vcc', '--version'], capture_output=True, text=True).stdout.strip()
    name = f'screen-borrow_{PRESET}.vcc'
    print(f'copying {os.path.getsize(vcc_file) / 1e9:.2f} GB to {dest}/{name} ...')
    shutil.copyfile(vcc_file, f'{dest}/{name}')
    with open(f'{dest}/README.md', 'w') as f:
        f.write(f"""# VCC 2026 prediction: screen-borrow {PRESET}

File: {name} ({os.path.getsize(vcc_file):,} bytes, sha256 {h.hexdigest()})
Built: {stamp} UTC with notebooks/submission_2026_colab.ipynb, build-only (not submitted from this run)
Code: KunaalAgarwal/virtual_cell_notebooks, branch {branch}, commit {commit}
vcc CLI: {cli}
Panel: {run.get('panel_id') or DATASET}, contexts {', '.join(map(str, run['contexts']))}; {run['n_cells']:,} cells ({run['cells_per_pert']} per target per context), {run['nnz']:,} stored counts
Preset: {run['preset']}, seed {run['seed']}
Parameters:
```
{json.dumps(run['params'], indent=1, default=str)}
```
Leaderboard model name used when this preset is submitted: "screen-borrow {PRESET}"
(the first leaderboard entry, preset c1_calibrated, was named "screen-borrow c1 calibrated").
""")
    shutil.copyfile(f'{WORK}/pred.json', f'{dest}/build_settings.json')
    drive.flush_and_unmount()
    print(f'saved to My Drive/{DRIVE_FOLDER}/{PRESET}_{stamp}/ . In Drive: right-click the folder, Share, General access: Anyone with the link.')


### Optional, High-RAM runtime only: the one-step official prep instead of `package_vcc.py`

In [ ]:
# !vcc prep {WORK}/pred.h5ad -g {WORK}/data/gene_names.csv --perts {WORK}/data/pert_counts.csv -o {WORK}/pred_official.vcc